In [ ]:
import os

import matplotlib.pyplot as plt
import cv2
import OpenEXR
import Imath
import numpy as np
from pprint import pprint
import random
from tqdm import tqdm
import time
import os

#os.environ["TF_GPU_THREAD_MODE"] = "gpu_private"

from pprint import pprint
import numpy as np
import tensorflow as tf
import keras
from keras.callbacks import Callback
from tensorflow.python.client import device_lib
from keras.applications import MobileNetV2
from keras.layers import Dense
from keras.models import Model
from keras.optimizers import Adam
# from keras.optimizers import SGD
from keras.utils import Sequence

# os.environ["TF_DATA_EXPERIMENTAL_OPTIMIZATION_AUTOTUNE_RAM_BUDGET"] = "18884901888"


gpu_options = tf.compat.v1.GPUOptions(per_process_gpu_memory_fraction=0.95)
sess = tf.compat.v1.Session(config=tf.compat.v1.ConfigProto(gpu_options=gpu_options))

# gpus = tf.config.experimental.list_physical_devices('GPU')
# if gpus:
#     try:
#         for gpu in gpus:
#             tf.config.experimental.set_memory_growth(gpu, True)
#         print("Memory growth enabled for GPUs")
#     except RuntimeError as e:
#         print(f"Error enabling memory growth: {e}")

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.test.is_gpu_available())

# Get detailed device information
print(device_lib.list_local_devices())

In [ ]:
def ordinal_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    return tf.reduce_mean(tf.square(y_true - y_pred),axis=1)

from keras.models import load_model


model_path='/mnt/Velocity_Vault/Autofocus/Model/Full_Trained.keras'


model=load_model(model_path,custom_objects={'ordinal_loss': ordinal_loss})

In [ ]:
def check_image_connection(image_path):
    # Display the "Reconnect Drive" message
    
    print("Reconnect Drive")
    checking=True
    
    while checking:
        # Try reading the image
        if (not os.path.isfile(image_path)) or (not os.path.isdir('/mnt/Velocity_Vault')):
            time.sleep(10)
        else:
            print(image_path)
            checking=False
    print("Drive Connected")

def convert_range(value, old_min=0, old_max=65535, new_min=-1, new_max=1):
    # Linear interpolation formula
    return ((value - old_min) / (old_max - old_min)) * (new_max - new_min) + new_min


def get_image_array(image_path,x,y,change_range=False):
    
    # Read the image in grayscale
    image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED) 
    
    if image is None:
        check_image_connection(image_path)
        image = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    
    send=image[x:x+128,y:y+128]
    
    if change_range:
        send=convert_range(send)
        
    #print(image.shape)
    
    return send


def load_depth_image(image_path,x,y):
    # Load standard image format (e.g., PNG, JPG)
    depth_array = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    
    if depth_array is None:
        check_image_connection(image_path)
        depth_array = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)

    # Convert to float32 for consistency
    depth_array = np.float32(depth_array)

    return depth_array[x:x+32, y:y+32]

def load_conf_image(exr_path, x,y,channel="R"):
    
    try:
        exr_file = OpenEXR.InputFile(exr_path)
    except Exception as e:
        check_image_connection(exr_path)
        exr_file = OpenEXR.InputFile(exr_path)
        
    header = exr_file.header()
    dw = header['dataWindow']
    width = dw.max.x - dw.min.x + 1
    height = dw.max.y - dw.min.y + 1
    pixel_type = Imath.PixelType(Imath.PixelType.FLOAT)  
    channel_data = exr_file.channel(channel, pixel_type)
    np_array = np.frombuffer(channel_data, dtype=np.float32).reshape((height, width))
    
    return np_array[x:x+32, y:y+32]


def approx_depth(depth, conf):
    
    depth_values = depth.flatten()
    confidence_values = conf.flatten()

    # Compute the weighted average depth
    weighted_sum = np.sum(depth_values * confidence_values)
    total_confidence = np.sum(confidence_values)
    
    approximate_depth = weighted_sum / total_confidence

    return approximate_depth


def find_slice(desc_list, number):
    if not desc_list:
        raise ValueError("The list cannot be empty.")
    if not all(desc_list[i] >= desc_list[i + 1] for i in range(len(desc_list) - 1)):
        raise ValueError("The list must be in descending order.")
    
    closest_index = min(range(len(desc_list)), key=lambda i: abs(desc_list[i] - number))
    return closest_index


def predict_slice(depth):
    
    approx=depth/255.0

    max_val=3.9
    min_val=0.1

    metre=(max_val * min_val) / (max_val - (max_val - min_val) * approx)
    metre*=1000

    slice_focal_length=[3910.92,2289.27,1508.71,1185.83,935.91,801.09,700.37,605.39,546.23,486.87,447.99,407.40,379.91,350.41,329.95,307.54,291.72,274.13,261.53,247.35,237.08,225.41,216.88,207.10,198.18,191.60,183.96,178.29,171.69,165.57,160.99,155.61,150.59,146.81,142.35,138.98,134.99,131.23,127.69,124.99,121.77,118.73,116.40,113.63,110.99,108.47,106.54,104.23,102.01]

    slice_focus=find_slice(slice_focal_length,metre)
    
    return slice_focus
    

def transpose_list(input_list):
    np_array = np.array(input_list)
    transposed_array = np.transpose(np_array, (1, 2, 0))
    return transposed_array

def make_stack(left_loc,right_loc,depth_loc,conf_loc,x,y):
    
    
    frames=np.zeros((98, 128, 128))
    
    for i in range(49):
        left_frame=left_loc+"/"+str(i)+'/result_up_pd_left_center.png'
        right_frame=right_loc+"/"+str(i)+'/result_up_pd_right_center.png'

        frames[i*2]=get_image_array(left_frame,x,y,change_range=True)
        frames[i*2+1]=get_image_array(right_frame,x,y,change_range=True)
        
    # display_map(frames[0])
        
    slices=transpose_list(frames)
        
    depth_frame=depth_loc+'/result_merged_depth_center.png'
    conf_frame=conf_loc+'/result_merged_conf_center.exr'
    
    x//=4
    y//=4
    
    depth=load_depth_image(depth_frame,x,y)
    conf=load_conf_image(conf_frame,x,y)
    
    approx=approx_depth(depth,conf)
    focus=predict_slice(approx)
    
        
        
    return slices,focus

In [ ]:
test_path='/mnt/Velocity_Vault/Autofocus/Test'
scene='/secondfloorgames_1'
patch_x=700
patch_y=500

left_loc='/mnt/Velocity_Vault/Autofocus/Test/raw_up_left_pd'+scene
right_loc='/mnt/Velocity_Vault/Autofocus/Test/raw_up_right_pd'+scene
depth_loc='/mnt/Velocity_Vault/Autofocus/Test/merged_depth'+scene
conf_loc='/mnt/Velocity_Vault/Autofocus/Test/merged_conf'+scene

slices,focus=make_stack(left_loc,right_loc,depth_loc,conf_loc,patch_x,patch_y)

test_dataset=[slices]
test_truth=[focus]

test_dataset=np.array(test_dataset)
test_truth=np.array(test_truth)


In [ ]:
print(test_dataset.shape)
print(test_truth.shape)

In [ ]:

predictions=model.predict(test_dataset)

print(predictions.shape)
pprint(predictions)

In [ ]:
def get_max_positions(array):
    # Find the index of the max value along each row
    return np.argmax(array, axis=1)

pred_truth=get_max_positions(predictions)

In [ ]:
def calculate_accuracy(array1, array2):
    # Check if both arrays are the same shape
    if array1.shape != array2.shape:
        raise ValueError("Arrays must have the same shape")
    
    # Compare the arrays and calculate the accuracy
    correct_predictions = np.sum(array1 == array2)
    total_elements = array1.size
    
    accuracy = correct_predictions / total_elements
    return accuracy*100

pred_test=calculate_accuracy(pred_truth,test_truth)

print("Exact Correctness \n")

print("\nPredicted Truth vs Ground Truth - ",pred_test)



In [ ]:
pprint(test_truth)
pprint(pred_truth)


In [ ]:
def calculate_deviation(predictions, ground_truth):
    assert len(predictions) == len(ground_truth), "Prediction and ground truth lists must have the same length."
    
    new_array = (ground_truth - predictions)
    ma=np.max(new_array)
    av=np.mean(new_array)
    
    # Compute squared errors
    squared_errors = [(p - gt) ** 2 for p, gt in zip(predictions, ground_truth)]
    
    # Calculate mean of squared errors
    mse = sum(squared_errors) / len(ground_truth)
    
    return ma,av,mse

ma,av,mse=calculate_deviation(pred_truth,test_truth)

print("Deviation Correctness - Prediction \n")

print('\nMax Deviation - ',ma)
print('\nAvg Deviation - ',av)
print('\nMean Square Deviation - ',mse)

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

def plot_occurrences(numbers):
    """
    Plots a horizontal bar graph of the occurrences of elements in a list.

    Parameters:
        numbers (list): A list of numbers to analyze.
    """
    # Count occurrences of each element
    occurrences = Counter(numbers)
    

    # Extract keys (unique numbers) and their corresponding values (counts)
    elements = list(occurrences.keys())
    counts = list(occurrences.values())
    print(np.mean(counts))

    # Plot a horizontal bar graph
    plt.figure(figsize=(8, 5))
    plt.barh(elements, counts, color="skyblue")
    plt.xlabel("Occurrences")
    plt.ylabel("Elements")
    plt.title("Occurrences of Elements in the List")
    plt.grid(axis="x", linestyle="--", alpha=0.7)
    plt.show()
    
def plot_frame(pred):
    """
    Plots a horizontal bar graph of the occurrences of elements in a list.

    Parameters:
        numbers (list): A list of numbers to analyze.
    """
    # Count occurrences of each element
    
    ele=[i for i in range(len(pred))]
    # Plot a horizontal bar graph
    plt.figure(figsize=(8, 5))
    plt.barh(ele, pred, color="skyblue")
    plt.xlabel("Occurrences")
    plt.ylabel("Elements")
    plt.title("Occurrences of Elements in the List")
    plt.grid(axis="x", linestyle="--", alpha=0.7)
    plt.show()

plot_frame(predictions[0])
print(predictions)
# print(np.sum(predictions, axis=1))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import matplotlib.patches as patches

def display_image_grid(image_paths, pred=None, grnd=None, x=0, y=0):
    
    # Validate input size
    if len(image_paths) != 49:
        raise ValueError("The input array must contain exactly 49 image paths.")
    if pred is not None and (pred < 0 or pred >= 49):
        raise ValueError("The 'pred' index must be between 0 and 48.")
    if grnd is not None and (grnd < 0 or grnd >= 49):
        raise ValueError("The 'grnd' index must be between 0 and 48.")
    
    # Create a figure with 7x7 grid
    fig, axes = plt.subplots(7, 7, figsize=(14, 14))
    fig.subplots_adjust(hspace=0.1, wspace=0.1)
    
    # Iterate through each image and its respective subplot
    for idx, ax in enumerate(axes.flat):
        # Load the image
        img = Image.open(image_paths[idx])
        
        # Crop the image as [x:x+32, y:y+32]
        cropped_img = img.crop((x, y, x + 32, y + 32))
        
        # Display the cropped image
        ax.imshow(cropped_img)
        ax.axis('off')
        
        # Add border for pred and grnd indices
        if idx == pred:
            rect = patches.Rectangle(
                (0, 0), 1, 1, linewidth=7, edgecolor='red', facecolor='none',
                transform=ax.transAxes, clip_on=False
            )
            ax.add_patch(rect)
        if idx == grnd:
            rect = patches.Rectangle(
                (0, 0), 1, 1, linewidth=7, edgecolor='blue', facecolor='none',
                transform=ax.transAxes, clip_on=False
            )
            ax.add_patch(rect)
    
    # Display the grid
    plt.show()



In [ ]:

display=[]
for i in range(49):
    frame=str(i)+"/"
    display.append(test_path+'/scaled_images'+scene+'/'+str(i)+'/result_scaled_image_center.jpg')

display_image_grid(display,pred=pred_truth[0],grnd=test_truth[0],x=patch_x//4,y=patch_y//4)